# Program 04: Elastic Weight Consolidation (EWC)

**What:** program 03 plus EWC, a penalty for changing weights that were important for old tasks.

**How:** after each task, estimate the diagonal Fisher information (average squared gradient of the log-likelihood, per weight) and store a copy of the weights. While training later tasks, add λ·Σ F_i·(θ_i − θ_old_i)² to the loss.

**Why:** EWC is the classic regularization approach. The Fisher idea comes back in unlearning methods like SSD, which use the same "how important is this weight?" measure to decide which weights to *damage*.

**Understanding note:** some weights matter more than others for a task. Measuring that importance lets you protect knowledge, or later target it for removal.

**Done when:** forgetting is clearly lower than in program 03. Expect EWC to help less in class-incremental learning; note why.

## 1. Setup
Same settings as program 03 (5 epochs per task, lr 0.03, batch 32, seed 0), so the only difference is EWC.

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import common/

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Subset

from common.data import N_TASKS, SplitCIFAR100, task_classes
from common.models import resnet18_cifar
from common.train import average_accuracy, evaluate_tasks, forgetting, new_acc_matrix, train_one_epoch
from common.utils import RESULTS_DIR, get_device, log_run, set_seed

SETTINGS = {
    "epochs": 5, "lr": 0.03, "batch_size": 32, "seed": 0, "workers": 2,
    "ewc_lambda": 100.0,     # strength of the penalty (see section 4)
    "ewc_gamma": 1.0,        # how much old Fisher is kept each task (1.0 = keep all)
    "fisher_samples": 1000,  # training images used to estimate the Fisher, per task
}
set_seed(SETTINGS["seed"])
device = get_device()
print("device:", device)

## 2. Data and model
Same as program 03: one ResNet-18, one 100-way output, class-incremental.

In [ ]:
data = SplitCIFAR100(seed=SETTINGS["seed"], batch_size=SETTINGS["batch_size"], num_workers=SETTINGS["workers"])
model = resnet18_cifar(num_classes=100).to(device)
scaler = torch.amp.GradScaler(enabled=device.type == "cuda")

## 3. The Fisher information: how important is each weight?

**Idea.** Take one training image, compute the loss −log p(correct label | image), and backprop. The gradient of a weight says how much the loss would change if that weight moved a little.

- Large gradient → moving this weight hurts the task a lot → **important**.
- Gradient near 0 → the task doesn't care about this weight → **free to change**.

Gradients can be + or −, so we **square** them, then **average over many images**. That average is the (diagonal, empirical) Fisher information F_i, one number per weight, the same shape as the model.

Details:
- **One image at a time** (batch size 1). Averaging gradients over a batch *first* would let + and − cancel before squaring, and underestimate importance.
- **`model.eval()`** so BatchNorm uses its stored statistics (a single image has no useful batch statistics).
- **Plain images** (no random crop/flip): we want the importance for the real data, not augmented copies.
- We use the **true label**. This is the "empirical Fisher", the simplest version and the one most code uses.
- 1,000 images per task is enough for a stable estimate and takes about 10 to 20 seconds.

In [ ]:
def compute_fisher(model, task, n_samples):
    """Average squared gradient of -log p(y|x) for each weight, over n_samples images of one task."""
    model.eval()
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters()}

    # Plain (non-augmented) training images of this task, a random sample of n_samples.
    indices = data._indices(task_classes(task), train=True)
    pick = np.random.choice(indices, size=min(n_samples, len(indices)), replace=False)
    subset = Subset(data.train_set_plain, pick.tolist())

    for image, label in subset:
        image = image.unsqueeze(0).to(device)          # batch of 1
        label = torch.tensor([label], device=device)
        model.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(image), label)    # = -log p(label | image)
        loss.backward()
        for n, p in model.named_parameters():
            if p.grad is not None:
                fisher[n] += p.grad.detach() ** 2

    model.zero_grad(set_to_none=True)
    return {n: f / len(subset) for n, f in fisher.items()}

## 4. The EWC penalty

After task t we store two things:
- **θ\***: a copy of the weights right after task t (the "good for old tasks" point).
- **F**: the Fisher, i.e. how important each weight is.

While training the next tasks we add:

$$\text{penalty} = \lambda \sum_i F_i \, (\theta_i - \theta^*_i)^2$$

Think of it as a **spring** attached to every weight, pulling it back to θ\*. F_i is the stiffness of that spring: important weights get stiff springs, unimportant ones almost none. So the network can still learn the new task, but mostly by moving weights the old tasks didn't need.

**Online EWC.** With 10 tasks, storing a separate F and θ\* per task costs 10 copies of the model. Instead we keep **one running Fisher**: `F = γ·F_old + F_new`, and θ\* = the weights after the latest task. With γ = 1.0, every old task's importance is kept in full.

**Choosing λ.** The Fisher values are tiny (often 1e-6 to 1e-2), so λ has to be large enough to matter. 100 is a reasonable starting point. The training loop below prints the cross-entropy and the penalty separately:
- penalty much smaller than CE → spring too weak → raise λ (try 1,000 or 10,000).
- new-task training accuracy far below program 03's ~55-75% → spring too stiff → lower λ.

In [ ]:
fisher, theta_star = None, None     # nothing to protect before task 0 is learned
penalty_log = []                    # penalty value of each batch, to check λ

def ewc_penalty(model, images, labels, logits):
    """Called by train_one_epoch on every batch and added to the cross-entropy."""
    if fisher is None:
        return torch.zeros((), device=device)
    total = 0.0
    for n, p in model.named_parameters():
        total = total + (fisher[n] * (p - theta_star[n]) ** 2).sum()
    penalty = SETTINGS["ewc_lambda"] * total
    penalty_log.append(penalty.item())
    return penalty

## 5. Train task by task with EWC

Same loop as program 03, with two additions:
1. `extra_loss=ewc_penalty` adds the spring to every batch.
2. After each task, update the Fisher and θ\*.

Note: `train_one_epoch` reports the **total** loss (CE + penalty). We print the average penalty separately so you can see how big each part is. About 1 to 2 minutes per task.

In [ ]:
acc_cil, acc_til = new_acc_matrix(), new_acc_matrix()
start = time.time()
for task in range(N_TASKS):
    train_loader, _ = data.get_task_loaders(task)
    optimizer = torch.optim.SGD(model.parameters(), lr=SETTINGS["lr"], momentum=0.9)
    for epoch in range(SETTINGS["epochs"]):
        penalty_log.clear()
        loss, train_acc = train_one_epoch(model, train_loader, optimizer, scaler, device, extra_loss=ewc_penalty)
    avg_pen = float(np.mean(penalty_log)) if penalty_log else 0.0

    # Remember this task: add its Fisher to the running total, store the weights.
    new_f = compute_fisher(model, task, SETTINGS["fisher_samples"])
    fisher = new_f if fisher is None else {n: SETTINGS["ewc_gamma"] * fisher[n] + new_f[n] for n in fisher}
    theta_star = {n: p.detach().clone() for n, p in model.named_parameters()}

    cil, til = evaluate_tasks(model, data, task, device)
    acc_cil[task, : task + 1] = cil
    acc_til[task, : task + 1] = til
    print(f"task {task}: train acc {train_acc*100:5.1f}%  loss {loss:.3f} (penalty {avg_pen:.3f})  "
          f"({time.time()-start:.0f}s)\n   class-IL: " + " ".join(f"{a*100:5.1f}" for a in cil) +
          "\n   task-IL:  " + " ".join(f"{a*100:5.1f}" for a in til))

## 6. Read the accuracy matrices

In [ ]:
def show(acc, title):
    print(f"{title} (%)")
    print("      " + "".join(f"{t:>6}" for t in range(N_TASKS)) + "   avg")
    for i in range(N_TASKS):
        cells = "".join("     ." if np.isnan(v) else f"{v*100:6.1f}" for v in acc[i])
        print(f"after {i}{cells}  {average_accuracy(acc, i)*100:5.1f}")
    print()

show(acc_cil, "EWC, class-IL")
show(acc_til, "EWC, task-IL")

## 7. Compare with program 03

This loads the matrices program 03 saved (`results/03_acc_matrix.npz`). If the file is missing, run program 03 first.

**Done when:** forgetting is clearly lower than in program 03.

In [ ]:
naive = np.load(RESULTS_DIR / "03_acc_matrix.npz")
n_cil, n_til = naive["class_il"], naive["task_il"]
last = N_TASKS - 1

print(f"{'':22}{'naive (03)':>12}{'EWC (04)':>12}")
for name, a, b in [("final avg, class-IL", average_accuracy(n_cil, last), average_accuracy(acc_cil, last)),
                   ("final avg, task-IL",  average_accuracy(n_til, last), average_accuracy(acc_til, last)),
                   ("forgetting, class-IL", forgetting(n_cil), forgetting(acc_cil)),
                   ("forgetting, task-IL",  forgetting(n_til), forgetting(acc_til))]:
    print(f"{name:22}{a*100:11.1f}%{b*100:11.1f}%")

x = np.arange(N_TASKS)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (a_n, a_e, title) in zip(axes[:2], [(n_cil, acc_cil, "class-IL"), (n_til, acc_til, "task-IL")]):
    ax.plot(x, [average_accuracy(a_n, i) * 100 for i in x], marker="o", label="naive (03)")
    ax.plot(x, [average_accuracy(a_e, i) * 100 for i in x], marker="o", label="EWC (04)")
    ax.set(title=f"Average accuracy over seen tasks, {title}", xlabel="after training task",
           ylabel="accuracy (%)", ylim=(0, 100), xticks=x)
    ax.legend()

ax = axes[2]
ax.plot(x, n_til[:, 0] * 100, marker="o", label="naive, task-IL")
ax.plot(x, acc_til[:, 0] * 100, marker="o", label="EWC, task-IL")
ax.plot(x, n_cil[:, 0] * 100, marker="o", ls="--", label="naive, class-IL")
ax.plot(x, acc_cil[:, 0] * 100, marker="o", ls="--", label="EWC, class-IL")
ax.set(title="Task 0 accuracy over time", xlabel="after training task", ylabel="accuracy (%)",
       ylim=(-2, 100), xticks=x)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "04_ewc_vs_naive.png", dpi=120)
plt.show()

## 8. Why EWC helps less in class-IL

EWC protects **weights**. But program 03 showed that class-IL fails mainly because of **output bias**: training task t pushes every other class's logit down, so the newest classes always win.

EWC doesn't fix that well:
- The output weights of *future* classes have Fisher ≈ 0 for old tasks (old tasks never used them), so nothing stops them from growing large.
- The Fisher is measured on one task's images alone. It never measured "old class vs new class", because new classes didn't exist yet.

Below we repeat program 03's section 6 check: what does the model predict for task 0 images now? Then we look at **where** the Fisher is large, layer by layer, to see which parts of the network EWC protects most.

In [ ]:
_, test0 = data.get_task_loaders(0)
model.eval()
with torch.no_grad():
    preds = torch.cat([model(x.to(device)).argmax(1).cpu() for x, _ in test0])
pred_task = (preds // 10).bincount(minlength=N_TASKS)
print("task 0 test images, predicted as a class of task:")
for t in range(N_TASKS):
    print(f"  task {t}: {pred_task[t].item()/len(preds)*100:5.1f}%")

w = model.fc.weight.detach().norm(dim=1).cpu().view(N_TASKS, 10).mean(1)
print("\naverage output-weight norm per task:", " ".join(f"{v:.2f}" for v in w))

# Mean Fisher per layer group: which parts does EWC hold in place most?
groups = ["conv1", "layer1", "layer2", "layer3", "layer4", "fc"]
means = [torch.cat([f.flatten() for n, f in fisher.items() if n.startswith(g)]).mean().item() for g in groups]
plt.figure(figsize=(6, 3.5))
plt.bar(groups, means)
plt.yscale("log")
plt.title("Mean Fisher per layer group (after all tasks)")
plt.ylabel("mean F (log scale)")
plt.tight_layout()
plt.show()

## 9. Optional: try other λ values

One run per λ, about 15 minutes each. Only do this if section 5's penalty looked far too small or too large, or if you want to see the trade-off: bigger λ protects old tasks more but makes new tasks harder to learn (lower diagonal).

To try one, change `SETTINGS["ewc_lambda"]` in section 1 and run the notebook again from the top. Every run is logged, so you can compare them in `results/runs.jsonl`.

## 10. Log the run

In [ ]:
to_json = lambda a: [[None if np.isnan(v) else round(float(v), 4) for v in r] for r in a]
log_run("04_ewc", SETTINGS,
        {"acc_class_il": to_json(acc_cil), "acc_task_il": to_json(acc_til),
         "final_avg_class_il": average_accuracy(acc_cil, N_TASKS - 1),
         "final_avg_task_il": average_accuracy(acc_til, N_TASKS - 1),
         "forgetting_class_il": forgetting(acc_cil), "forgetting_task_il": forgetting(acc_til),
         "minutes": round((time.time() - start) / 60, 1)})
np.savez(RESULTS_DIR / "04_acc_matrix.npz", class_il=acc_cil, task_il=acc_til)
print("logged to", RESULTS_DIR / "runs.jsonl")

## Lab notebook

*Write one paragraph here: what you saw, and what surprised you.*

Questions to answer in it:
1. How much did EWC reduce forgetting compared with program 03, in class-IL and in task-IL? Which improved more?
2. Are task 0 images still predicted as the newest task's classes? Why can't a penalty on weights stop that?
3. Which layers have the largest Fisher? What does that say about which parts of the network each task depends on?
4. The Fisher says which weights matter for a task. How could you use the same numbers to *remove* a class instead of protecting it? (This is the idea behind SSD, an unlearning method.)